# 04 — Particle Swarm Optimization (PSO) Feature Selection

Binary PSO searches for a compact feature mask. This notebook uses the shared experiment pipeline so all optimizers receive the same data splits, classifiers, seeds, and evaluation rules.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Particle Swarm Optimization

In [2]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bpso(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
    w=0.4,
    c1=2.05,
    c2=2.05,
):
    # Initialize particles
    positions = np.random.randint(0, 2, (pop_size, n_features))
    velocities = np.random.uniform(-1, 1, (pop_size, n_features))

    # Personal best
    pbest_positions = positions.copy()
    pbest_scores = np.array([obj_func(p) for p in positions])

    # Global best
    best_idx = np.argmin(pbest_scores)
    gbest_position = pbest_positions[best_idx].copy()
    gbest_score = pbest_scores[best_idx]

    convergence = []

    for _ in range(iterations):

        for i in range(pop_size):

            r1 = np.random.rand(n_features)
            r2 = np.random.rand(n_features)

            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - positions[i])
                + c2 * r2 * (gbest_position - positions[i])
            )

            probs = sigmoid(velocities[i])

            positions[i] = (np.random.rand(n_features) < probs).astype(int)

            # Prevent empty feature subset
            if positions[i].sum() == 0:
                positions[i, np.random.randint(n_features)] = 1

            score = obj_func(positions[i])

            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = positions[i].copy()

                if score < gbest_score:
                    gbest_score = score
                    gbest_position = positions[i].copy()

        convergence.append(gbest_score)

    return gbest_position, gbest_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def pso_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bpso(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
        w=0.4,
        c1=2.05,
        c2=2.05,
    )


pso_results = run_feature_selector("PSO", pso_runner)
pso_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'random_forest', 0)


Saved: ('breast', 'xgboost', 0)


Saved: ('heart', 'svm', 0)


Saved: ('heart', 'random_forest', 0)


Saved: ('heart', 'xgboost', 0)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
1,breast,random_forest,PSO,0,0.031053,0.982456,1.000000,0.952381,0.975610,0.998347,15,5.344559,0.253998,"[""radius_mean"", ""perimeter_mean"", ""smoothness_...",results/smoke/artifacts/breast__random_forest_...,results/smoke/artifacts/breast__random_forest_...
2,breast,xgboost,PSO,0,0.030719,0.938596,0.906977,0.928571,0.917647,0.992394,14,2.372455,0.085569,"[""perimeter_mean"", ""area_mean"", ""compactness_m...",results/smoke/artifacts/breast__xgboost__pso__...,results/smoke/artifacts/breast__xgboost__pso__...
3,heart,svm,PSO,0,0.160433,0.815217,0.846939,0.813725,0.830000,0.867767,11,0.444112,0.023720,"[""trestbps"", ""oldpeak"", ""sex_Female"", ""cp_asym...",results/smoke/artifacts/heart__svm__pso__seed0...,results/smoke/artifacts/heart__svm__pso__seed0...
4,heart,random_forest,PSO,0,0.166213,0.771739,0.812500,0.764706,0.787879,0.838176,12,6.662472,0.323602,"[""trestbps"", ""chol"", ""thalch"", ""oldpeak"", ""cp_...",results/smoke/artifacts/heart__random_forest__...,results/smoke/artifacts/heart__random_forest__...
5,heart,xgboost,PSO,0,0.157452,0.788043,0.831579,0.774510,0.802030,0.894189,17,1.944824,0.088350,"[""age"", ""trestbps"", ""chol"", ""thalch"", ""ca"", ""s...",results/smoke/artifacts/heart__xgboost__pso__s...,results/smoke/artifacts/heart__xgboost__pso__s...
